<a href="https://colab.research.google.com/github/Maheepkaur03/Code_AI/blob/main/News2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pandas faiss-cpu sentence-transformers openpyxl textblob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 90.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from textblob import TextBlob
from collections import Counter

# ---- STEP 1: Load the data ----
# Replace with your filename (Excel or CSV supported)
df = pd.read_excel("newsdata.xlsx")
headlines = df["Title"].astype(str).tolist()
categories = df["category_name"].astype(str).tolist()

# ---- STEP 2: Load Sentence Embedding Model ----
# Use 'BAAI/bge-large-en-v1.5' for sentence embeddings.
model = SentenceTransformer('BAAI/bge-large-en-v1.5')

# ---- STEP 3: Generate Embeddings (Normalize for cosine similarity) ----
embeddings = model.encode(headlines, normalize_embeddings=True)
embeddings = np.array(embeddings, dtype="float32")

# ---- STEP 4: Build FAISS Index (Cosine similarity, so use IndexFlatIP with normalized vectors) ----
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

# ---- STEP 5: Prepare Data Mapping ----
headline_data = [{"headline": h, "category": c} for h, c in zip(headlines, categories)]

# ---- STEP 6: Utility: Sentiment Analysis ----
def get_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0:
        return "Positive"
    elif polarity < 0:
        return "Negative"
    else:
        return "Neutral"

# ---- STEP 7: Query Function ----
def categorize_headline(input_headline, k=5):
    # Embed user input
    query_emb = model.encode([input_headline], normalize_embeddings=True)
    query_emb = np.array(query_emb, dtype="float32")
    # Search
    D, I = index.search(query_emb, k)
    neighbor_categories = [headline_data[i]['category'] for i in I[0]]
    predicted_category = Counter(neighbor_categories).most_common(1)[0][0]
    sentiment = get_sentiment(input_headline)
    similar_headlines = [headline_data[i]['headline'] for i in I[0]]

    result = {
        "input_headline": input_headline,
        "sentiment": sentiment,
        "predicted_category": predicted_category,
        "similar_headlines": similar_headlines
    }
    return result

# ---- STEP 8: Example Usage ----
# Test with a new headline
test_headline = "Pharma company obtains regulatory approval for breakthrough drug"
result = categorize_headline(test_headline)
print(f"Headline: {result['input_headline']}")
print(f"Sentiment: {result['sentiment']}")
print(f"Predicted Category: {result['predicted_category']}\n")
print("Most Similar Headlines:")
for h in result['similar_headlines']:
    print(" -", h)

# ---- STEP 9: Save/Reload Index & Data (optional) ----
# import pickle
# faiss.write_index(index, "news_headlines_gemma.index")
# with open("headline_data_gemma.pkl", "wb") as f:
#     pickle.dump(headline_data, f)

# To later reload:
# index = faiss.read_index("news_headlines_gemma.index")
# with open("headline_data_gemma.pkl", "rb") as f:
#     headline_data = pickle.load(f)

# ---- STEP 10: User input loop ----
while True:
    user_headline = input("\nEnter a news headline (or type 'exit' to quit):\n> ").strip()
    if user_headline.lower() == "exit":
        print("Exiting.")
        break

    sentiment = get_sentiment(user_headline)

    query_emb = model.encode([user_headline], normalize_embeddings=True).astype("float32")
    D, I = index.search(query_emb, 5)
    neighbor_categories = [headline_data[i]['category'] for i in I[0]]
    predicted_category = Counter(neighbor_categories).most_common(1)[0][0]
    similar_headlines = [headline_data[i]['headline'] for i in I[0]]

    print("\n--- Analysis ---")
    print("Sentiment:", sentiment)
    print("Predicted category:", predicted_category)
    print("\nMost similar headlines from your data:")
    for h, c in zip(similar_headlines, neighbor_categories):
        print(f"  - [{c}] {h}")

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Headline: Pharma company obtains regulatory approval for breakthrough drug
Sentiment: Neutral
Predicted Category: NoCategory

Most Similar Headlines:
 - ASMS :Bartronics India Limited has Submitted to the Exchange a copy of Disclosure under Regulation 31(4) of the Securities and Exchange Board of India (Substantial Acquisition of Shares and Takeovers) Regulations, 2011.
 - ASMS :Bartronics India Limited has Submitted to the Exchange a copy of Disclosure under Regulation 31(4) of the Securities and Exchange Board of India (Substantial Acquisition of Shares and Takeovers) Regulations, 2011.
 - ASMS :Bartronics India Limited has informed the Exchange about Certificate under SEBI (Depositories and Participants) Regulations, 2018
 - ASMS :Bartronics India Limited has informed the Exchange about Certificate under SEBI (Depositories and Participants) Regulations, 2018
 - ASMS :Bartronics India Limited has informed the Exchange regarding change in Registered Office of the company.

--- Analysi

In [3]:
# 1. Install dependencies
!pip install --quiet pandas faiss-cpu sentence-transformers openpyxl textblob
import nltk
nltk.download('punkt')

# 2. Imports
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from textblob import TextBlob
from collections import Counter

# 3. Load your Excel file (update path/filename if needed)
df = pd.read_excel("newsdata.xlsx")  # or 'news.xlsx'
headlines = df["Title"].astype(str).tolist()
categories = df["category_name"].astype(str).tolist()

headline_data = [{"headline": h, "category": c} for h, c in zip(headlines, categories)]

# 4. Sentiment cache for fast filtering
def get_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0:
        return "Positive"
    elif polarity < 0:
        return "Negative"
    else:
        return "Neutral"

all_sentiments = [get_sentiment(h) for h in headlines]

# 5. Load embedding model and FAISS index for semantic Q&A
model = SentenceTransformer('BAAI/bge-large-en-v1.5')
embeddings = model.encode(headlines, normalize_embeddings=True)
embeddings = np.array(embeddings, dtype="float32")
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("Chatbot is ready! Type your query or type 'exit' to quit.\n")

# 6. Chatbot loop
while True:
    user_input = input("\nYou: ").strip()
    if user_input.lower() == "exit":
        print("Exiting chatbot.")
        break

    # Category-based retrieval
    if "list of" in user_input.lower() and "category" in user_input.lower():
        parts = user_input.lower().split("category")
        if len(parts) > 1:
            cat_query = parts[1].replace("headlines", "").strip().title()
            found = [h for h, c in zip(headlines, categories) if cat_query in c]
            if found:
                print(f"\nFound {len(found)} headlines in '{cat_query}' category:")
                for i, headline in enumerate(found, 1):
                    print(f"{i}. {headline}")
            else:
                print(f"No headlines found in '{cat_query}' category.")

    # Sentiment-based retrieval
    elif "list of" in user_input.lower() and "headlines" in user_input.lower() and ("positive" in user_input.lower() or "negative" in user_input.lower() or "neutral" in user_input.lower()):
        if "positive" in user_input.lower():
            target_sentiment = "Positive"
        elif "negative" in user_input.lower():
            target_sentiment = "Negative"
        else:
            target_sentiment = "Neutral"
        found = [h for h, s in zip(headlines, all_sentiments) if s == target_sentiment]
        if found:
            print(f"\nFound {len(found)} {target_sentiment.lower()} headlines:")
            for i, headline in enumerate(found, 1):
                print(f"{i}. {headline}")
        else:
            print(f"No {target_sentiment.lower()} headlines found.")

    # Free-text category queries (e.g., "Show me ESG news", "Headlines about Legal")
    elif any(kw in user_input.lower() for kw in (c.lower() for c in set(categories))):
        lower_input = user_input.lower()
        found_any = False
        for cat in set(categories):
            if cat.lower() in lower_input:
                found = [h for h, c in zip(headlines, categories) if c.lower() == cat.lower()]
                if found:
                    print(f"\nFound {len(found)} headlines about '{cat}':")
                    for i, headline in enumerate(found, 1):
                        print(f"{i}. {headline}")
                    found_any = True
        if not found_any:
            print("No matching category found in query.")

    # Semantic search for similar headlines
    else:
        print("Performing semantic search for related headlines...")
        q_emb = model.encode([user_input], normalize_embeddings=True).astype("float32")
        D, I = index.search(q_emb, 5)
        print("\nMost similar headlines:")
        for rank, idx in enumerate(I[0], 1):
            print(f"{rank}. [{categories[idx]}] {headlines[idx]}")

        # Also output predicted category (majority neighbor vote)
        neighbor_cats = [categories[idx] for idx in I[0]]
        pred_cat = Counter(neighbor_cats).most_common(1)[0][0]
        print(f"Predicted category: {pred_cat}")

        # Also output sentiment of user input
        print(f"Sentiment: {get_sentiment(user_input)}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Chatbot is ready! Type your query or type 'exit' to quit.


Found 199 headlines in '' category:
1. ED, Gurugram Zonal Office has provisionally attached movable and immovable properties valuing Rs. 557.49 Crore on 26.03.2025 under the provisions of PMLA, 2002 in the case M/s Amtek Auto Limited, M/s ARG Limited, M/s ACIL limited, M/s Metalyst Forging Limited and M/s Castex Technologies Limited, Arvind Dham, promoter Amtek Group and others. The attachment follows the Provisional Attachment of Rs. 5115.31 Crore dated 05.09.2024 issued by ED that has been confirmed by the PMLA Adjudicating Authority.-Enforcement Directorate
2. ED, Gurugram Zonal Office has provisionally attached movable and immovable properties valuing Rs. 557.49 Crore on 26.03.2025 under the provisions of PMLA, 2002 in the case M/s Amtek Auto Limited, M/s ARG Limited, M/s ACIL limited, M/s Metalyst Forging Limited and M/s Castex Technologies Limited, Arvind Dham, promoter Amtek Group and others. The attachment follows the 

In [4]:
!wget -qO- --compression=auto --timeout=5 https://ollama.com/install.sh | sh

!ollama serve > server.log 2>&1 &
!ollama pull gemma3:12b

!sudo apt install poppler-utils

import os

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
ollama
ollama

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.9).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.


In [5]:
!pip install ollama

In [6]:
!ollama pull llama3

In [7]:
!pip install --upgrade langchain langchain-core langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.0 MB/s eta 0:00:00


In [12]:
!ollama pull mxbai-embed-large
!ollama pull llama3

In [13]:
!pip install pandas faiss-cpu openpyxl textblob langchain-community ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 61.0 MB/s eta 0:00:00


In [24]:
# --- Install required packages if needed ---
# pip install pandas faiss-cpu openpyxl textblob langchain-community ollama

import pandas as pd
import numpy as np
from textblob import TextBlob
from langchain.vectorstores import FAISS
from langchain.embeddings import OllamaEmbeddings
from langchain.docstore.document import Document
from langchain.llms import Ollama
from langchain.chains import ConversationalRetrievalChain
from langchain_core.prompts import PromptTemplate # Import PromptTemplate

# === STEP 1: Load and preprocess your Excel data ===
df = pd.read_excel("Newsdata2.xlsx")
df = df.rename(columns={'Title': 'headline', 'category_name': 'category'})

# Add sentiment if missing, or compute with TextBlob
def get_sentiment(text):
    try:
        polarity = TextBlob(str(text)).sentiment.polarity
        if polarity > 0.1:
            return "positive"
        elif polarity < -0.1:
            return "negative"
        else:
            return "neutral"
    except:
        return "neutral"

if 'sentiment' not in df.columns or df['sentiment'].isnull().any():
    df['sentiment'] = df['headline'].astype(str).apply(get_sentiment)

# Clean strings and fill missing columns to avoid errors
for col in ['headline', 'category', 'sentiment', 'CompanyName', 'Last_Fetch']:
    if col not in df.columns:
        df[col] = ""
    df[col] = df[col].astype(str).str.strip()

# Prepare unique lists for quick lookups and LLM context
unique_categories = sorted({c for c in df['category'] if c})
unique_companies = sorted({c for c in df['CompanyName'] if c})
unique_dates = sorted({c for c in df['Last_Fetch'] if c})

# === STEP 2: Create Documents with rich content and metadata ===
documents = []
for _, row in df.iterrows():
    headline = row['headline']
    category = row['category']
    sentiment = row['sentiment']
    company = row['CompanyName']
    date = row['Last_Fetch']

    tags = ", ".join(set([category.lower(), sentiment.lower(), company.lower(), date.lower()]))

    content = (
        f"Headline: {headline}\n"
        f"Category: {category}\n"
        f"Sentiment: {sentiment}\n"
        f"Company: {company}\n"
        f"Date: {date}\n"
        f"Tags: {tags}"
    )
    metadata = {
        "category": category,
        "sentiment": sentiment,
        "company": company,
        "date": date,
        "tags": tags
    }
    documents.append(Document(page_content=content, metadata=metadata))

# === STEP 3: Build FAISS Vector Store with Ollama Embeddings ===
print("🔍 Generating embeddings and building FAISS index...")
embeddings = OllamaEmbeddings(model="mxbai-embed-large")  # change model if you prefer
db = FAISS.from_documents(documents, embeddings)
print(f"Total documents stored: {len(db.docstore._dict)}")

# === STEP 4: Setup Ollama LLM and ConversationalRetrievalChain ===
llm = Ollama(model="llama3")  # or replace with your preferred Ollama LLM

retriever = db.as_retriever(search_kwargs={"k": 40})

# Optional system prompt to improve responses
system_prompt_template = """
You are a helpful news assistant. You have access to these fields:
Headline, Category, Sentiment, Company, Date.
Known categories: {categories}
Known companies: {companies}
Known dates: {dates}
Answer user queries using these fields and format tabular or list outputs when appropriate.

Chat History:
{chat_history}

Context:
{context}

Question: {question}
"""

system_prompt = PromptTemplate(
    template=system_prompt_template,
    input_variables=["chat_history", "context", "question"],
    partial_variables={
        "categories": ", ".join(unique_categories),
        "companies": ", ".join(unique_companies[:10]) + ('...' if len(unique_companies) > 10 else ''),
        "dates": ", ".join(unique_dates[:5]) + ('...' if len(unique_dates) > 5 else '')
    }
)


qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    # Move system_prompt to combine_docs_chain_kwargs
    combine_docs_chain_kwargs={"prompt": system_prompt},
    # llm_kwargs={"system_prompt": system_prompt}, # Remove this line
)

# === STEP 5: Chat loop ===
chat_history = []
print("\n🤖 NewsBot is ready with enhanced retrieval! Type 'exit' to quit.\n")

while True:
    query = input("You: ").strip()
    if query.lower() in {"exit", "quit"}:
        print("👋 Exiting NewsBot. Goodbye.")
        break

    # Quick interception for list queries to avoid LLM call & errors
    normalized = query.lower()
    if normalized in {"list all categories", "show categories", "list of headline categories"}:
        print("Categories:\n", ", ".join(unique_categories))
        continue
    if normalized in {"list all companies", "show companies"}:
        print("Companies:\n", ", ".join(unique_companies[:50]))
        continue
    if normalized in {"list all dates", "show dates"}:
        print("Dates:\n", ", ".join(unique_dates))
        continue

    # Correct usage: pass 'question' key to qa_chain.invoke()
    result = qa_chain.invoke({
        "question": query,
        "chat_history": chat_history
    })

    print(f"\n🤖 Bot:\n{result['answer']}\n")

    # Append to chat history as HumanMessage and AIMessage objects
    chat_history.append(HumanMessage(content=query))
    chat_history.append(AIMessage(content=result["answer"]))

🔍 Generating embeddings and building FAISS index...
Total documents stored: 542

🤖 NewsBot is ready with enhanced retrieval! Type 'exit' to quit.

You: list all categories
Categories:
 Adverse Media, Business Transactions, Capital Market, Contracts, Disruption, ESG, Earnings, Estimates, Exchange Disclosures, Expansion, Financial Crime, Financial Distress, Hot Stocks, Legal, M & A, Management Change, Market Commentary, NoCategory, Opportunities, Partnerships, Press Release, Product Launch, Ratings, Recalls, Regulatory, Tax
You: list headlines with positive sentiment

🤖 Bot:
Based on the provided data, here are the headlines with a positive sentiment:

1. Bonus Alert: Patanjali Foods declares first ever free share issue; Details here - CNBC TV18 (Sentiment: positive)
2. ED attaches fresh assets of Amtek Group worth Rs 557 cr (Multiple entries) (Sentiment: positive)
3. Q4 Results Live: Balrampur Chini Mills Profit Meet Estimates, Patanjali Foods Profit Up 74% (Sentiment: positive)
4. Pata